# 04: Labeling

**DATA 5500, Senior Design**
**Sara Choque Molina**

One hundred articles were drawn at random from the clean corpus and labeled independently by three people: the student and two outside reviewers with no background in the subject. This notebook measures how much they agreed, decides each article by majority, and records which rules caused the disagreements.

Each reviewer worked in their own copy so that nobody could see anyone else's answers. Those copies were then consolidated into the single file this notebook reads, `labels_review_done.xlsx`, which is the record of the round.

## The rules being tested

The full rule set is in `docs/codebook.md`. This is the short version, which is what the annotators applied.

Every article gets exactly one of three labels. The decision is two questions, in order:

1. **Did someone make a threat, and does the article report it as current news?** If not, the label is `unrelated`.
2. **Was the threat aimed at a named school?** If yes, `targeted_threat`. If no, `nontargeted_threat`.

| Label | Meaning |
|---|---|
| `targeted_threat` | A threat of violence against a named school, district or person at a school, including the lockdown, closure or arrest that followed it |
| `nontargeted_threat` | Student threats with no named target: the viral warning, rumors, schools adding security against a threat that named nobody, and analysis of the trend |
| `unrelated` | No threat of violence against a school: COVID closures, school board disputes, attacks with no threat reported beforehand, and the word threat used in a legal or figurative sense |

Four rules decide the hard cases. A threat aimed at a named school stays `targeted_threat` even when it turns out to be false. An attack that already happened is not a threat. The label follows the extracted text rather than the link. One article carries one label, however many threats it describes.

## Block 1: Load the labels

**This block handles the input.** It opens the consolidated file and checks that every article carries a label from all three annotators, since a blank cell cannot be compared.

In [5]:
import os

import numpy as np
import pandas as pd

FOLDER = r"C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases"

REVIEW_FILE = os.path.join(FOLDER, "labels_review_done.xlsx")

ANNOTATORS = ["mine", "reviewer2", "reviewer3"]
LABELS = ["targeted_threat", "nontargeted_threat", "unrelated"]

review = pd.read_excel(REVIEW_FILE).set_index("id")

print("Articles:", len(review))
for name in ANNOTATORS:
    print(f"  {name:10} {review[name].notna().sum()} labeled, {review[name].isna().sum()} blank")

# An article missing any label cannot be compared, so the agreement is measured on the
# articles all three scored.
scored = review.dropna(subset=ANNOTATORS).copy()
print()
print("Articles labeled by all three:", len(scored))

Articles: 100
  mine       100 labeled, 0 blank
  reviewer2  100 labeled, 0 blank
  reviewer3  100 labeled, 0 blank

Articles labeled by all three: 100


## Block 2: How much the annotators agreed

**This block handles the agreement measure.** It turns the three sets of labels into one number for the codebook.

Percent agreement on its own is misleading, because two people picking at random would still match some of the time. Kappa subtracts that chance agreement, so it answers how much they agreed beyond luck.

Cohen kappa compares two annotators at a time. Fleiss kappa takes all three at once and is the number reported for the codebook. On the usual scale, 0.41 to 0.60 is moderate agreement and 0.61 to 0.80 is substantial.

In [6]:
import itertools
from collections import Counter


def cohen_kappa(a, b):
    """Agreement between two annotators, corrected for agreement by chance."""
    n = len(a)
    observed = (a == b).mean()

    # Chance agreement is how often they would match if each picked labels at random
    # with their own frequencies.
    count_a, count_b = Counter(a), Counter(b)
    expected = sum(count_a[lab] / n * count_b[lab] / n for lab in LABELS)

    return observed, (observed - expected) / (1 - expected)


print("Pairwise agreement")
for one, two in itertools.combinations(ANNOTATORS, 2):
    observed, kappa = cohen_kappa(scored[one].values, scored[two].values)
    print(f"  {one:10} vs {two:10}  {observed:.0%} agreement   Cohen kappa {kappa:.3f}")

# Fleiss kappa takes all three at once. votes holds, for every article, how many of the
# three annotators chose each label.
n_raters = len(ANNOTATORS)
votes = np.zeros((len(scored), len(LABELS)))
for row, (_, article) in enumerate(scored.iterrows()):
    for name in ANNOTATORS:
        votes[row, LABELS.index(article[name])] += 1

# Share of annotator pairs that agreed, per article, then averaged over the articles.
per_article = ((votes ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
observed = per_article.mean()

# Chance agreement, from how often each label was used across the whole set.
label_share = votes.sum(axis=0) / (len(scored) * n_raters)
expected = (label_share ** 2).sum()

fleiss = (observed - expected) / (1 - expected)

print()
print(f"Fleiss kappa over all three annotators: {fleiss:.3f}")

Pairwise agreement
  mine       vs reviewer2   69% agreement   Cohen kappa 0.516
  mine       vs reviewer3   72% agreement   Cohen kappa 0.558
  reviewer2  vs reviewer3   78% agreement   Cohen kappa 0.652

Fleiss kappa over all three annotators: 0.575


## Block 3: The agreed answer and where the disagreements are

**This block handles the decision.** It settles each article by majority and shows which pairs of labels were confused.

Where at least two annotators chose the same label, that label becomes the agreed answer. The instructor recommended an odd number of annotators for this reason, so a tie is only possible when all three differ.

Which pairs of labels get mixed up matters more than the total. A boundary that two labels keep crossing is a rule the codebook has not stated sharply enough.

In [7]:
# How many of the three picked the most common label for each article.
top = votes.max(axis=1)

print("All three agreed:   ", int((top == 3).sum()), f"({(top == 3).mean():.0%})")
print("Two of three agreed:", int((top == 2).sum()))
print("Three-way split:    ", int((top == 1).sum()))
print()


def majority(article):
    """Return the label at least two annotators chose, or nothing on a three-way split."""
    choice, count = Counter(article[ANNOTATORS]).most_common(1)[0]
    return choice if count >= 2 else None


review["majority"] = scored.apply(majority, axis=1)

all_same = (review["mine"] == review["reviewer2"]) & (review["mine"] == review["reviewer3"])
not_scored = review[ANNOTATORS].isna().any(axis=1)

review["agreement"] = np.where(
    not_scored, "not labeled",
    np.where(review["majority"].isna(), "no majority",
             np.where(all_same, "all three", "two of three")),
)

# Which pairs of labels get mixed up tells us which rule is unclear.
pairs = Counter()
for _, article in scored.iterrows():
    chosen = sorted({article[name] for name in ANNOTATORS})
    if len(chosen) > 1:
        pairs[" vs ".join(chosen)] += 1

print("Disagreements are")
for pair, n in pairs.most_common():
    print(f"  {pair:45} {n}")

print()
print("Agreed label")
print(review["majority"].value_counts(dropna=False).to_string())

All three agreed:    60 (60%)
Two of three agreed: 39
Three-way split:     1

Disagreements are
  nontargeted_threat vs targeted_threat         19
  targeted_threat vs unrelated                  14
  nontargeted_threat vs unrelated               6
  nontargeted_threat vs targeted_threat vs unrelated 1

Agreed label
majority
targeted_threat       48
unrelated             32
nontargeted_threat    19
None                   1


## Block 4: Save the result

**This block handles the record.** It writes the agreed answer and the agreement level back into the same file, so one workbook holds the three labels, the decision and the notes.

In [8]:
review.to_excel(REVIEW_FILE)
print("Saved:", REVIEW_FILE)
print()
print(review["agreement"].value_counts().to_string())

Saved: C:\Users\sara_\Desktop\College\DATA 5500-01 Senior Des\data bases\labels_review_done.xlsx

agreement
all three       60
two of three    39
no majority      1


## Round 1 conclusions

Three annotators labeled the same 100 articles independently.

**Fleiss kappa is the number to report.** On the usual scale, 0.41 to 0.60 is moderate agreement and 0.61 to 0.80 is substantial. The result sits below the 0.70 the project aimed for, which means the rules are workable but not yet sharp.

Half of the disagreements sit on one boundary: `targeted_threat` against `nontargeted_threat`. That is the question of whether a school named in an article was the one threatened or was only responding to a threat aimed at nobody in particular. It is the rule the codebook did not state clearly enough, and it is the rule that changes before the next round.

The notes point to a second problem that was not expected. Across the three annotators, at least eight of the 100 articles carry text that is not a news story: stray symbols, a weather listing, advertising copy. All three labeled those `unrelated` and agreed, but they are not unrelated articles, they are extraction failures that survived the cleaning. At that rate roughly 8% of the corpus is still unusable, and the repeated-line method did not catch them because each one is broken in its own way.

One more result is worth recording. The two outside reviewers, who had only the codebook, agreed with each other more than either agreed with the student. That suggests the student is applying criteria that are not written down, and that the codebook can be improved by stating them.

This round did what a pilot is for. It produced a measured number for the codebook, it located the single rule responsible for most of the disagreement, and it surfaced a data problem that rereading the rules would never have found.